In [1]:
from discrete_tools import *
from sympy import symbols, Poly, factorint, binomial, expand, Integer, simplify, factor, factorint, pretty, latex
from math import comb, factorial
import math
import sympy as sp

x, y = symbols('x y')

# Reference & Tools

## Derangements

$D_n$ = number of permutations where **no element** is in its original position.
$$D_n = n! \sum_{k=0}^{n} \frac{(-1)^k}{k!} \approx \frac{n!}{e}$$

In [2]:
# Count derangements
for n in range(1, 8):
    print(f"D({n}) = {derangements_count(n)}")

# List all derangements of a string
items = 'ABCD'
d = get_derangements(items)
print(f"\nDerangements of '{items}': {len(d)} total")
print(d)

# Check a specific permutation
original = 'ABCDE'
candidate = 'BAEDC'
is_d = all(candidate[i] != original[i] for i in range(len(original)))
print(f"\nIs '{candidate}' a derangement of '{original}'? {is_d}")

D(1) = 0
D(2) = 1
D(3) = 2
D(4) = 9
D(5) = 44
D(6) = 265
D(7) = 1854

Derangements of 'ABCD': 9 total
['BADC', 'BCDA', 'BDAC', 'CADB', 'CDAB', 'CDBA', 'DABC', 'DCAB', 'DCBA']

Is 'BAEDC' a derangement of 'ABCDE'? False


## N-cubes (Hypercube graphs $Q_n$)

$Q_n$ has $2^n$ vertices (all $n$-bit strings), $n \cdot 2^{n-1}$ edges,
is $n$-regular, bipartite, and has a Hamilton cycle (Gray code) for $n \geq 1$.

In [3]:
# Stats for Q_1 through Q_5
print(f"{'n':>3}  {'vertices':>9}  {'edges':>8}  {'degree':>7}  Hamilton")
print("-" * 40)
for n in range(1, 6):
    info = hypercube_info(n)
    print(f"{n:>3}  {info['vertices']:>9}  {info['edges']:>8}  {info['degree']:>7}  {info['hamiltonian']}")

# Show vertices and edges of Q_3
n = 4
info = hypercube_info(n)
print(f"\nQ_{n} vertices ({info['vertices']}):")
print(' ', info['vertex_list'])
print(f"Q_{n} edges ({info['edges']}):")
for e in hypercube_edges(n):
    print(f"  {e[0]}, {e[1]}")

# Bipartition of Q_n: vertices with even vs odd number of 1s
verts = hypercube_vertices(n)
even_part = [v for v in verts if v.count('1') % 2 == 0]
odd_part  = [v for v in verts if v.count('1') % 2 == 1]
print(f"\nBipartition of Q_{n}:")
print(f"  Even-parity: {even_part}")
print(f"  Odd-parity:  {odd_part}")

  n   vertices     edges   degree  Hamilton
----------------------------------------
  1          2         1        1  True
  2          4         4        2  True
  3          8        12        3  True
  4         16        32        4  True
  5         32        80        5  True

Q_4 vertices (16):
  ['0000', '0001', '0010', '0011', '0100', '0101', '0110', '0111', '1000', '1001', '1010', '1011', '1100', '1101', '1110', '1111']
Q_4 edges (32):
  0000, 0001
  0000, 0010
  0000, 0100
  0000, 1000
  0001, 0011
  0001, 0101
  0001, 1001
  0010, 0011
  0010, 0110
  0010, 1010
  0011, 0111
  0011, 1011
  0100, 0101
  0100, 0110
  0100, 1100
  0101, 0111
  0101, 1101
  0110, 0111
  0110, 1110
  0111, 1111
  1000, 1001
  1000, 1010
  1000, 1100
  1001, 1011
  1001, 1101
  1010, 1011
  1010, 1110
  1011, 1111
  1100, 1101
  1100, 1110
  1101, 1111
  1110, 1111

Bipartition of Q_4:
  Even-parity: ['0000', '0011', '0101', '0110', '1001', '1010', '1100', '1111']
  Odd-parity:  ['0001', '0010',

## Inclusion-Exclusion

$|A_1 \cup \cdots \cup A_n| = \sum|A_i| - \sum|A_i \cap A_j| + \cdots$

In [4]:
# Brute-force inclusion-exclusion over a list of sets
def union_size_by_inclusion_exclusion(sets):
    n = len(sets)
    total = 0
    for r in range(1, n + 1):
        for combo in combinations(range(n), r):
            inter = set.intersection(*(sets[i] for i in combo))
            total += ((-1) ** (r + 1)) * len(inter)
    return total

# Example: how many integers in [1,100] divisible by 2, 3, or 5?
U = set(range(1, 101))
A2 = {x for x in U if x % 2 == 0}
A3 = {x for x in U if x % 3 == 0}
A5 = {x for x in U if x % 5 == 0}

ie_result = union_size_by_inclusion_exclusion([A2, A3, A5])
direct = len(A2 | A3 | A5)
print(f"Divisible by 2, 3, or 5 in [1,100]: {ie_result} (direct: {direct})", "✓" if ie_result == direct else "✗")

Divisible by 2, 3, or 5 in [1,100]: 74 (direct: 74) ✓


## Pigeonhole Principle

If $n$ objects go into $k$ boxes, at least one box has $\lceil n/k \rceil$ objects.
**Template:** find the minimum $n$ to guarantee a pair/group with some property.

In [5]:
import math

def pigeonhole_min(k, guarantee_per_box):
    """Minimum objects needed so some box has >= guarantee_per_box objects."""
    return k * (guarantee_per_box - 1) + 1

# How many people needed to guarantee 3 share a birthday? (365 days)
k, g = 365, 3
print(f"Min people for 3 sharing a birthday: {pigeonhole_min(k, g)}")

# How many cards drawn from a deck to guarantee 2 of the same suit?
k, g = 4, 2
print(f"Min cards for 2 of same suit: {pigeonhole_min(k, g)}")

# Generalised: guarantee >= g items in one box among k boxes
for (k, g, desc) in [(7, 2, '2 born same weekday'), (12, 2, '2 born same month')]:
    print(f"Min for {desc}: {pigeonhole_min(k, g)}")

Min people for 3 sharing a birthday: 731
Min cards for 2 of same suit: 5
Min for 2 born same weekday: 8
Min for 2 born same month: 13


## Bit Strings with Constraints

In [6]:
from itertools import product as iproduct

def bit_strings(n):
    return [''.join(str(b) for b in bits) for bits in iproduct([0,1], repeat=n)]

n = 5
all_bs = bit_strings(n)
print(f"Total {n}-bit strings: {len(all_bs)}")

# Start with 1
start1 = [s for s in all_bs if s[0] == '1']
print(f"Start with 1: {len(start1)}")

# At least two consecutive 1s
consec = [s for s in all_bs if '11' in s]
print(f"Contain '11': {len(consec)}")

# Exactly k ones
k = 3
exact_k = [s for s in all_bs if s.count('1') == k]
print(f"Exactly {k} ones: {len(exact_k)}  (= C({n},{k}) = {comb(n,k)})")

Total 5-bit strings: 32
Start with 1: 16
Contain '11': 19
Exactly 3 ones: 10  (= C(5,3) = 10)


## Multiplicative Inverse & Euler's φ

In [7]:
# Multiplicative inverse: find x such that a*x ≡ 1 (mod m)
# Exists iff gcd(a, m) = 1
a, m = 7, 26
inv = pow(a, -1, m)   # Python 3.8+ built-in modular inverse
print(f"{a}^(-1) mod {m} = {inv}  (check: {a}*{inv} mod {m} = {a*inv % m})")

# Euler φ(n): count of integers in [1,n] coprime to n
print("\nEuler φ:")
for n in [6, 10, 12, 15, 30]:
    print(f"  φ({n}) = {sp.totient(n)}")

# φ of a prime power: φ(p^k) = p^(k-1)(p-1)
p, k = 7, 3
print(f"\nφ({p}^{k}) = {sp.totient(p**k)}  (formula: {p}^{k-1}·({p}-1) = {p**(k-1)*(p-1)})")

7^(-1) mod 26 = 15  (check: 7*15 mod 26 = 1)

Euler φ:
  φ(6) = 2
  φ(10) = 4
  φ(12) = 4
  φ(15) = 8
  φ(30) = 8

φ(7^3) = 294  (formula: 7^2·(7-1) = 294)


## GCD, LCM, and Extended Euclidean

In [8]:
a, b = 252, 198
print(f"gcd({a},{b}) = {math.gcd(a,b)}")
print(f"lcm({a},{b}) = {lcm(a,b)}")
print()
# Full EEA table with quotient and Bézout coefficients
extended_euclidean_table(a, b)

gcd(252,198) = 18
lcm(252,198) = 2772

  gcd(252, 198) = 18
  Bézout: 18 = (4)·252 + (-5)·198
  ↑ last non-zero in 'remainder' column


step,quotient,remainder,s (×252),t (×198)
i64,i64,i64,i64,i64
0,null,252,1,0
1,null,198,0,1
2,1,54,1,-1
3,3,36,-3,4
4,1,18,4,-5
5,2,0,null,null


## Primes

In [9]:
print("Primes below 100:", primes_below(100))
print(f"Is 101 prime? {is_prime(101)}")
print(f"Is 102 prime? {is_prime(102)}")

# Prime factorisation
n = 360
print(f"\n{n} = ", end='')
factors = sp.factorint(n)
print(' · '.join(f'{p}^{e}' if e > 1 else str(p) for p, e in sorted(factors.items())))
print(f"φ({n}) = {sp.totient(n)}")

Primes below 100: [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89, 97]
Is 101 prime? True
Is 102 prime? False

360 = 2^3 · 3^2 · 5
φ(360) = 96


## Set Expression Parser

Write set expressions as strings using: `inter`, `union`, `without` (set difference), `~` or `overline` (complement).
Lets you paste in the MC options directly and get ✓/✗ automatically.

In [10]:
# Use a concrete universe and named sets
U = set(range(10))
sets = {
    'A': {0, 2, 4, 6, 8},   # evens
    'B': {1, 2, 3, 4, 5},
    'C': {3, 4, 5, 6, 7},
    'U': U,
}

# Operators: inter  union  without (set diff)  ~  overline (complement)
target_expr = "A inter ~(B without C)"
target = evaluate_set_expression(target_expr, sets, U)
print(f"Target = A ∩ ~(B\\C) = {sorted(target)}\n")

# Check a list of MC option expressions, the correct one gets ✓
options = [
    "(A inter ~B) union (A inter C)",   # correct by De Morgan
    "A inter ~B inter C",               # wrong
    "(A union B) inter (A union ~C)",   # wrong
    "A inter ~B union C",               # wrong (precedence)
]
for opt in options:
    result = evaluate_set_expression(opt, sets, U)
    print(f"  {'✓' if result == target else '✗'}  {opt}")
    print(f"       = {sorted(result)}")

# find_equivalent_set_options does the same in one call:
print()
matches = find_equivalent_set_options(target_expr, options, sets, U)
print("Matching options:", matches)

Target = A ∩ ~(B\C) = [0, 4, 6, 8]

  ✓  (A inter ~B) union (A inter C)
       = [0, 4, 6, 8]
  ✗  A inter ~B inter C
       = [6]
  ✗  (A union B) inter (A union ~C)
       = [0, 1, 2, 4, 6, 8]
  ✗  A inter ~B union C
       = [0, 3, 4, 5, 6, 7, 8]

Matching options: ['(A inter ~B) union (A inter C)']


## Tautology Checker

Supported operators: `not`/`¬`/`!`, `and`/`∧`, `or`/`∨`, `=>`/`->`/`→`, `<=>`/`↔`
Variables: any single letter or word.

In [11]:
exprs = [
    "(p => q) or (not q => not p)",
    "(not p or not q) => (not (p and q))",
    "p <=> q",
    "(p or q or r) and (p or not q or not r)",
    "(p => q) <=> (not q => not p)",  # contrapositive, should be tautology
]

for expr in exprs:
    r = tautology_checker(expr)
    mark = '✓ tautology' if r['is_tautology'] else '✗ not tautology'
    print(f"{mark}  {expr}")

# Full truth table for a specific expression
print()
r = tautology_checker("(p => q) <=> (not p or q)")
print("Truth table for (p→q) ↔ (¬p∨q):")
for row in r['truth_table']:
    vals = ', '.join(f"{k}={int(v)}" for k, v in row['valuation'].items())
    print(f"  {vals}  →  {row['result']}")

✗ not tautology  (p => q) or (not q => not p)
✓ tautology  (not p or not q) => (not (p and q))
✗ not tautology  p <=> q
✗ not tautology  (p or q or r) and (p or not q or not r)
✓ tautology  (p => q) <=> (not q => not p)

Truth table for (p→q) ↔ (¬p∨q):
  p=0, q=0  →  True
  p=0, q=1  →  True
  p=1, q=0  →  True
  p=1, q=1  →  True


## Relation Classifier

Give it a set `S` and a relation `R` (set of pairs), it tells you reflexive/symmetric/etc.
and classifies as partial order, equivalence, Hasse diagram, well order, or none.

In [12]:
S = {1, 2, 3, 4}

# Divisibility on {1,2,3,4}: a R b iff a divides b
R_div = {(a, b) for a in S for b in S if b % a == 0}
print("Divisibility on {1,2,3,4}:", sorted(R_div))
c = classify_relation(S, R_div)
print("  Reflexive:", c['reflexive'], " Antisymmetric:", c['antisymmetric'],
      " Transitive:", c['transitive'])
print("  =>", c['classification'])

print()
# Congruence mod 2: a R b iff (a-b) divisible by 2
R_mod2 = {(a, b) for a in S for b in S if (a - b) % 2 == 0}
print("≡ mod 2 on {1,2,3,4}:", sorted(R_mod2))
c2 = classify_relation(S, R_mod2)
print("  Symmetric:", c2['symmetric'], " Transitive:", c2['transitive'])
print("  =>", c2['classification'])

Divisibility on {1,2,3,4}: [(1, 1), (1, 2), (1, 3), (1, 4), (2, 2), (2, 4), (3, 3), (4, 4)]
  Reflexive: True  Antisymmetric: True  Transitive: True
  => ['Partial order']

≡ mod 2 on {1,2,3,4}: [(1, 1), (1, 3), (2, 2), (2, 4), (3, 1), (3, 3), (4, 2), (4, 4)]
  Symmetric: True  Transitive: True
  => ['Equivalence relation']


## Function Classifier

`classify_function_finite(domain, codomain, func)`, tests well-definedness,
injectivity, surjectivity on a finite sample.

`classify_linear_function_descriptor(str)`, parses `'f: NN -> NN, f(x)=2x+7'` for linear functions.

In [13]:
# TODO: what if functions have domains like zz^+ or similair?

# Template: swap in your function and domain/codomain sample
functions = [
    ("f(x) = 2x+7  (NN→NN)",  range(0,20), range(0,50),  lambda x: 2*x+7),
    ("f(x) = x²    (NN→NN)",  range(0,10), range(0,100), lambda x: x**2),
    ("f(x) = x//2  (NN→NN)",  range(0,10), range(0,10),  lambda x: x//2),
    ("f(x) = x mod 3 (NN→{0,1,2})", range(0,20), [0,1,2], lambda x: x % 3),
]

for label, dom, cod, func in functions:
    r = classify_function_finite(dom, cod, func)
    print(f"{label:35s}  {r['classification']}")

# String descriptor for linear functions (parses the text directly)
print()
for desc in [
    "f: NN -> NN, f(x) = 2x + 7",
    "f: ZZ -> ZZ, f(x) = 3x + 1",
    "f: RR -> RR, f(x) = 5x - 2",
]:
    r = classify_linear_function_descriptor(desc)
    print(f"{desc:35s}  {r['classification']}")

f(x) = 2x+7  (NN→NN)                 Injective but not surjective
f(x) = x²    (NN→NN)                 Injective but not surjective
f(x) = x//2  (NN→NN)                 Well defined but neither surjective nor injective
f(x) = x mod 3 (NN→{0,1,2})          Surjective but not injective

f: NN -> NN, f(x) = 2x + 7           Injective but not surjective
f: ZZ -> ZZ, f(x) = 3x + 1           Injective but not surjective
f: RR -> RR, f(x) = 5x - 2           Both surjective and injective


## Polynomial GCD & Extended Euclidean

In [14]:
# gcd only
p1 = Poly(x**3 - 1, x)
p2 = Poly(x**3 + 2*x**2 + 2*x + 1, x)
g = polynomial_gcd(p1, p2)
print("gcd:", g.as_expr())

# Bézout identity, s*p1 + t*p2 = gcd
g, s, t = polynomial_extended_gcd(p1, p2, show_steps=True)

# Full EEA table with quotient column
print()
polynomial_extended_euclidean_table(p1, p2)

gcd: x**2 + x + 1
gcd = x**2 + x + 1
Bezout: (-1/2)*(x**3 - 1) + (1/2)*(x**3 + 2*x**2 + 2*x + 1)

  gcd = x**2 + x + 1
  Bézout: (-1/2)·(x**3 - 1) + (1/2)·(x**3 + 2*x**2 + 2*x + 1)
  ↑ last non-zero in 'remainder' column (made monic)


step,quotient,remainder,s (×f),t (×g)
i64,str,str,str,str
0,null,"""x**3 - 1""","""1""","""0"""
1,null,"""x**3 + 2*x**2 + 2*x + 1""","""0""","""1"""
2,"""1""","""-2*x**2 - 2*x - 2""","""1""","""-1"""
3,"""-x/2 - 1/2""","""0""",null,null


## Coefficient of a Monomial + Match MC Options

Expand a polynomial and extract the coefficient of $x^a y^b$,
then check which MC answer option it matches.

In [15]:
# Option strings must use sympy-parseable syntax.
# Use  binomial(n, k)  for C(n,k), and  **  for exponents.

poly = expand((2*x**3 - y**4)**10)
target_powers = {'x': 15, 'y': 20}

coeff = coefficient_of_monomial(poly, target_powers)
print(f"Coefficient of x^15·y^20 in (2x³-y⁴)^10: {coeff}\n")

# MC options, write them as sympy expressions
options = [
    "-binomial(10,5)*2**5",   # correct
    "binomial(10,5)*2**15",
    "-binomial(10,5)*2**15",
    "0",
    "-binomial(10,5)*2**10",
    "-binomial(10,3)*2**5",
]
matches = coefficient_matches_options(poly, target_powers, options)
print("Matching options:", matches)
print()

# Template: swap in your polynomial and powers
# poly  = expand((x**a - c*y**b)**n)
# coeff = coefficient_of_monomial(poly, {'x': target_x, 'y': target_y})
# coefficient_matches_options(poly, {'x': target_x, 'y': target_y}, your_options)

Coefficient of x^15·y^20 in (2x³-y⁴)^10: -8064

Matching options: ['-binomial(10,5)*2**5']



## System of Congruences (CRT)

Solves $x \equiv a_i \pmod{n_i}$ for pairwise coprime moduli.
Input: list of `[a, n]` pairs.

In [16]:
# Template, swap in your values
system = [
    [1, 2],   # x ≡ 1 (mod 2)
    [1, 5],   # x ≡ 1 (mod 5)
    [7, 9],   # x ≡ 7 (mod 9)
]
x0, m = congruences_system_solver(system)
print(f"Verify: x0={x0} satisfies each congruence?")
for a, n in system:
    print(f"  {x0} mod {n} = {x0 % n}  (need {a})  {'✓' if x0 % n == a else '✗'}")

{61 + 90k | k ∈ ℤ}
Verify: x0=61 satisfies each congruence?
  61 mod 2 = 1  (need 1)  ✓
  61 mod 5 = 1  (need 1)  ✓
  61 mod 9 = 7  (need 7)  ✓


## Permutation Constraints, All Options

| Parameter | Meaning |
|---|---|
| `must_contain_all` | all listed substrings must appear |
| `must_contain_any` | at least one listed substring must appear |
| `must_not_contain_any` | none of the listed substrings may appear |
| `exactly_one_of` | exactly one listed substring appears |
| `count_permutations_subsequence` | counts non-contiguous subsequence matches |

In [17]:
items = 'ABCDE'
total = len(get_permutations(items))
print(f"Total permutations of '{items}': {total}\n")

cases = [
    ("Contain AB (contiguous)",           dict(must_contain_all=['AB'])),
    ("Contain AB or CD",                   dict(must_contain_any=['AB','CD'])),
    ("Contain none of AB, BC, CD",         dict(must_not_contain_any=['AB','BC','CD'])),
    ("Contain exactly one of AB, CD",      dict(exactly_one_of=['AB','CD'])),
    ("Contain both AB and CD",             dict(must_contain_all=['AB','CD'])),
]
for label, kwargs in cases:
    count = count_permutations_with_constraints(items, **kwargs)
    print(f"  {count:4d}  {label}")

print()
for sub in ['AB','ACE','AE']:
    count = count_permutations_subsequence(items, sub)
    print(f"  {count:4d}  '{sub}' as subsequence")

Total permutations of 'ABCDE': 120

    24  Contain AB (contiguous)
    42  Contain AB or CD
    64  Contain none of AB, BC, CD
    36  Contain exactly one of AB, CD
     6  Contain both AB and CD

    60  'AB' as subsequence
    20  'ACE' as subsequence
    60  'AE' as subsequence


## Division with Remainder

In [18]:
# Integer division: a = q·b + r
divide_with_remainder(17, 5)
divide_with_remainder(100, 7)

17 = 3·5 + 2   (remainder 2)
100 = 14·7 + 2   (remainder 2)


(14, 2)

## r-Permutations, r-Combinations & All Subsets

In [19]:
items = 'ABCD'

# All 2-permutations (order matters)
print("P(4,2):", get_r_permutations(items, 2))

# All 2-combinations (order doesn't matter)
print("C(4,2):", get_r_combinations(items, 2))

# All non-empty subsets
print("All subsets:", get_combinations(items))

P(4,2): ['AB', 'AC', 'AD', 'BA', 'BC', 'BD', 'CA', 'CB', 'CD', 'DA', 'DB', 'DC']
C(4,2): ['AB', 'AC', 'AD', 'BC', 'BD', 'CD']
All subsets: ['A', 'B', 'C', 'D', 'AB', 'AC', 'AD', 'BC', 'BD', 'CD', 'ABC', 'ABD', 'ACD', 'BCD', 'ABCD']


## Minimum Selections to Guarantee a Pair (Pigeonhole)

In [20]:
# How many items must you draw to guarantee a pair summing to target?
# Example: numbers = {1,..,9}, guarantee a pair summing to 10
numbers = list(range(1, 10))
target = 10
n = minimum_selections_for_sum(numbers, target)
print(f"Minimum draws to guarantee a pair summing to {target}: {n}")

Minimum draws to guarantee a pair summing to 10: 6


## Linear Congruences with Coefficients (ax ≡ c mod n)

In [21]:
# solve_system_congruences handles ax ≡ c (mod n) — each entry: (coeff, rhs, modulus)
# Example: 2x ≡ 4 (mod 6)  AND  3x ≡ 9 (mod 12)
sol = solve_system_congruences([(2, 4, 6), (3, 9, 12)])
print("Solution:", sol)

NameError: name 'solve_system_congruences' is not defined

## Math Statement → Python Lambda

Use this when writing `lambda` predicates for brute-force option checkers.

### Divisibility

| Math | Python |
|---|---|
| $a \mid b$ | `b % a == 0` |
| $a \nmid b$ | `b % a != 0` |

### Congruence

| Math | Python |
|---|---|
| $a \equiv b \pmod{m}$ | `(a - b) % m == 0` |
| $a \not\equiv b \pmod{m}$ | `(a - b) % m != 0` |
| $a \equiv b \pmod{cm}$ | `(a - b) % (c*m) == 0` |

**Rule:** $a \equiv b \pmod{m}$ always means $m \mid (a - b)$. Expand the expressions first, then the $3$s (or whatever constants) cancel.

### Set membership

| Math | Python |
|---|---|
| $x \in \{km : k \in \mathbb{Z}\}$ | `x % m == 0` |
| $\gcd(a, b) = 1$ | `gcd(a, b) == 1` |
| $\gcd(a, b) \mid c$ | `c % gcd(a, b) == 0` |

### Logic connectives

| Math | Python |
|---|---|
| if $P$ then $Q$ | `not P or Q` |
| $P$ and $Q$ | `P and Q` |
| not $P$ | `not P` |
| "$P$ if $\gcd(c,m)=1$" | `gcd(c,m) != 1 or P` |

### Worked example

Statement: $14a \equiv 14b + 7m \pmod{m}$

Step 1 — apply definition: $m \mid (14a - (14b + 7m))$

Step 2 — simplify inside: $14a - 14b - 7m = 14(a-b) - 7m$

Step 3 — write it: `(14*(a-b) - 7*m) % m == 0`

# E25 Exam

## Q1, Euler's Totient of $pq$

If $p, q$ prime, $100 < p < q$, how many positive integers less than $pq$ are relatively prime to $pq$?

**Answer:** $pq - p - q + 1 = \varphi(pq)$

**Why:** $\varphi(pq) = (p-1)(q-1) = pq - p - q + 1$.

In [ ]:
# Verify numerically with any pair of primes > 100
p, q = 101, 103 # MUST BE PRIMES > 100
phi = sp.totient(p * q)
formula = p * q - p - q + 1
print(f"phi({p}*{q}) = {phi}")
print(f"pq - p - q + 1 = {formula}")
print("Match:", phi == formula)

# Check which answer option matches symbolically
options = ["p*q - p - q + 1", "p*q - 1", "p*q - p - q - 1"]
print("\nOption check:", check_totient_options(options))

## Q2, Nested Modular Arithmetic

Compute $(4^{100} \bmod 6)^{100} \bmod 10$.

**Answer:** $6$

In [ ]:
# the expression
expr = (4**100 % 6)**100 % 10

print(f"{expr}")

## Q3, Subsets of $\{1, \ldots, 99\}$ (Multi-part)

| Sub-question | Answer |
|---|---|
| How many subsets have 49 elements? | $\binom{99}{49}$ |
| How many subsets have odd # even AND even # odd? | $2^{97}$ |
| How many subsets have odd # odd AND even # even? | $2^{97}$ |
| How many subsets have an odd number of odd numbers? | $2^{98}$ |

In [ ]:
n = 99  # set is {1, 2, ..., 99}
m = 49  # subset size

# Exact size
print(f"Subsets of size 49: C({n}, {m}) = {comb(n, m)}")

# Parity counts
counts = subset_parity_counts_1_to_n(n)
print(f"\nOdd #{'{'}even{'}'}  AND  even #{'{'}odd{'}'}: {counts['odd_even_and_even_odd']}")
print(f"{pretty(factorint(counts['odd_even_and_even_odd'], visual=True))}")

print(f"\nOdd #{'{'}odd{'}'}  AND  even #{'{'}even{'}'}: {counts['odd_odd_and_even_even']}")
print(f"{pretty(factorint(counts['odd_odd_and_even_even'], visual=True))}")

print(f"\nOdd #{'{'}odd{'}'} (any #{'{'}even{'}'}): {counts['odd_odd']}")
print(f"{pretty(factorint(counts['odd_odd'], visual=True))}")

## Q4, System of Congruences (CRT)

$$x \equiv 1 \pmod{2}, \quad x \equiv 1 \pmod{5}, \quad x \equiv 7 \pmod{9}$$

**Answer:** $\{61 + 90k \mid k \in \mathbb{Z}\}$

In [ ]:
# Each pair is [a, n] meaning x ≡ a (mod n).
# Requires pairwise coprime moduli.
congruences_system_solver([
    [1, 2],
    [1, 5],
    [7, 9],
])

## Q5, GCD of Polynomials

Find $\gcd(x^3 - 1,\; x^3 + 2x^2 + 2x + 1)$.

**Answer:** $x^2 + x + 1$

In [ ]:
p1 = Poly(x**3 - 1, x)
p2 = Poly(x**3 + 2*x**2 + 2*x + 1, x)

g = polynomial_gcd(p1, p2)
print("GCD:", g.as_expr())

# Full EEA table
polynomial_extended_euclidean_table(p1, p2)

## Q6, Proof by Induction

**Claim:** $f(n) = \sum_{k=0}^{n} k \cdot k! = (n+1)! - 1$ for all $n \in \mathbb{N}$

**Correct fragment order: F, B, H, D**

### Why each fragment is chosen

**F -- base case (not E):**
The domain is $\mathbb{N} = \{0, 1, 2, \ldots\}$, so the proof must start at $n = 0$.
Fragment E uses $n = 1$ as the base case, leaving $n = 0$ unproven.
Fragment F verifies $f(0) = 0 \cdot 0! = 0$ and $(0+1)! - 1 = 1 - 1 = 0$. ✓

**B -- induction hypothesis (not A or C):**
- **A** has the direction backwards: assumes the $n+1$ case to prove the $n$ case.
- **C** says "assume for *all* $n \in \mathbb{N}$" -- this assumes exactly what we are trying to prove, making the argument circular.
- **B** correctly assumes the formula holds for *some fixed* $n$, then proves it for $n + 1$.

**H -- induction step (not G):**
Fragment G goes backwards: it starts from $f(n)$ and uses $f(n+1)$ to reach a conclusion about $f(n)$, which is circular.
Fragment H proceeds correctly by expanding $f(n+1)$:

$$f(n+1) = \sum_{k=0}^{n+1} k \cdot k! = \underbrace{\sum_{k=0}^{n} k \cdot k!}_{f(n)} + (n+1)(n+1)!$$

$$\stackrel{\text{IH}}{=} \bigl[(n+1)! - 1\bigr] + (n+1)(n+1)! = (n+1)!\bigl[1 + (n+1)\bigr] - 1 = (n+2)! - 1 \checkmark$$

**D -- conclusion:** States that the claim follows by the principle of mathematical induction.

*The verify cell below confirms the formula numerically; the fragment ordering itself requires logical reasoning.*

In [ ]:
# Verify the formula f(n) = (n+1)! - 1
def f(n):
    return sum(k * factorial(k) for k in range(n + 1))

formula = lambda n: factorial(n + 1) - 1

# Step 1: Check the base case
base_n = 0
print(f"Base case n={base_n}: f({base_n})={f(base_n)}, formula={formula(base_n)}, OK={f(base_n)==formula(base_n)}")

# Step 2: Check induction step, for each n, assuming formula(n) holds,
# verify that f(n+1) == formula(n+1).
# (This doesn't *prove* it, but catches wrong formulas fast.)
print("\nInduction step spot-checks:")
all_ok = True
for n in range(0, 8):
    lhs = f(n + 1)
    rhs = formula(n + 1)
    ok = lhs == rhs
    all_ok = all_ok and ok
    print(f"  n={n}: f(n+1)={lhs}, formula(n+1)={rhs}, ✓" if ok else f"  n={n}: MISMATCH f(n+1)={lhs} != {rhs}")

print(f"\n{'✓ Formula matches for all tested n, proof structure is plausible.' if all_ok else '✗ Formula is wrong.'}")

# Guide: to test YOUR candidate formula, replace `formula` above and re-run.
# E.g. to test a wrong candidate:
wrong_formula = lambda n: factorial(n + 1)
wrong_ok = all(f(n) == wrong_formula(n) for n in range(8))
print(f"\nWrong formula (n+1)! matches? {wrong_ok}")

## Q7, Piecewise Recursive Sequence

$$a_0=2,\; a_1=3,\; a_n = \begin{cases} a_{n-1}+n & \text{if } n \text{ even} \\ a_{n-1}+2a_{n-2} & \text{if } n \text{ odd} \end{cases}$$

Compute $a_5$.

**Answer:** $37$

In [ ]:
result = recursive_piecewise_value(
    n=5,
    initial_values={0: 2, 1: 3},
    even_rule=lambda n, a: a(n - 1) + n,
    odd_rule=lambda n, a: a(n - 1) + 2 * a(n - 2),
)
print(f"a_5 = {result}")

# Step-by-step trace
def a(n):
    if n == 0: return 2
    if n == 1: return 3
    if n % 2 == 0: return a(n-1) + n
    return a(n-1) + 2*a(n-2)

for i in range(6):
    print(f"  a_{i} = {a(i)}")

## Q8, Bipartite Graph Degree Sequences (Multi-part)

Bipartition $(V_1, V_2)$ with $|V_1|=4$, $|V_2|=5$.

| Degrees $V_1$ | Degrees $V_2$ | Answer |
|---|---|---|
| $1,2,2,2$ | $1,2,2,2,2$ | Does not exist, but will if we increase a degree in $V_1$ |
| $5,5,5,5$ | $4,4,4,4,4$ | Exists without multiple edges |
| $4,4,4,4$ | $5,5,5,5,5$ | Does not exist, but will if we increase a degree in $V_1$ |

In [ ]:
cases = [
    ([1, 2, 2, 2], [1, 2, 2, 2, 2]),   # sum(V1)=7 < sum(V2)=9  → increase V1
    ([5, 5, 5, 5], [4, 4, 4, 4, 4]),   # sums equal, Gale-Ryser OK → simple graph exists
    ([4, 4, 4, 4], [5, 5, 5, 5, 5]),   # sum(V1)=16 < sum(V2)=25 → increase V1
    ([5, 5, 5, 5], [2, 2, 2, 2, 2]),   # sum(V1)=20 > sum(V2)=10 → increase V2
]

for v1, v2 in cases:
    r = bipartite_degree_check(v1, v2)
    print(f"V1={v1}, V2={v2}")
    print(f"  sum(V1)={r['sum_V1']}, sum(V2)={r['sum_V2']}")
    print(f"  => {r['classification']}\n")

## Q9, Vandermonde Identity Lower Bound

$$\binom{m+n}{r} = \sum_{k=a}^{r} \binom{m}{r-k}\binom{n}{k}, \quad 0 < m < r < n$$

**Answer:** $a = r - m$

### Derivation

The full Vandermonde identity sums from $k = 0$ to $k = r$, but many terms are zero.
The goal is to find the smallest $k$ that gives a non-zero term.

$\binom{m}{r-k} = 0$ when the top is smaller than the bottom, i.e. when $r - k > m$, equivalently $k < r - m$.

$\binom{n}{k} = 0$ when $k > n$, but since $r < n$ the upper limit $k = r$ is already safe.

So the first $k$ that makes $\binom{m}{r-k}$ non-zero is $k = r - m$.

Since $r > m > 0$ we have $r - m > 0$, so the sum genuinely starts above $k = 0$.
The non-zero range is $r - m \leq k \leq r$, giving exactly $m + 1$ terms.

In [ ]:
# Vandermonde identity: C(m+n, r) = Σ_{k=a}^{r} C(m, r-k)·C(n, k)  where a = r-m
# In Python, a generator expression reads just like the sigma notation:
#
#   Σ_{k=a}^{r} C(m, r-k)·C(n, k)
#   sum( comb(m, r-k) * comb(n, k)  for k in range(a, r+1) )
#    ↑Σ     ↑ C(m, r-k)    ↑ C(n,k)          ↑ k from a to r

test_cases = [(2, 5, 3), (3, 7, 5), (1, 4, 2)]

for m, n, r in test_cases:
    a = r - m                          # lower bound of summation
    # Left hand side: C(m+n, r)
    lhs = comb(m + n, r)

    # Right hand side: Σ(k=a..r) C(m, r-k)·C(n, k)
    rhs = sum(comb(m, r - k) * comb(n, k) for k in range(a, r + 1))

    status = '✓' if lhs == rhs else '✗'
    print(f"{status} m={m}, n={n}, r={r}: C({m+n},{r}) = Σ(k={a}..{r}) C({m},{r}-k)·C({n},k) = {rhs}")

## Q10, Set Operations

$A=\{0,1,2,4\}$, $B=\{0,1,3,5\}$, $C=\{0,2,3,6\}$, $U=\{0,\ldots,7\}$

Which set equals $((A \cap B) \setminus C) \cup ((B \cap C) \setminus A) \cup ((C \cap A) \setminus B)$?

**Answer:** $\{1, 2, 3\}$

In [ ]:
U = set(range(8)) # 0 indexed, so if 0-7 you do range(8) to include 7
sets = {'A': {0,1,2,4}, 'B': {0,1,3,5}, 'C': {0,2,3,6}, 'U': U}

target_expr = "(A inter B) without C union (B inter C) without A union (C inter A) without B"


result = evaluate_set_expression(target_expr, sets, U)
print("Result:", sorted(result))


## Q11, Predicate Logic Translation

**Statement:** "For every positive rational $x$ there are positive integers $a$ and $b$ such that $x = a/b$ and $\gcd(a, b) = 1$."

**Answer:** $\forall x \in \mathbb{Q}^+\; \exists a \in \mathbb{Z}^+\; \exists b \in \mathbb{Z}^+\; (x = a/b \wedge G(a,b))$

### Why each part is what it is

| Component | Choice | Reason |
|---|---|---|
| $x$ | $\forall$ | The property must hold for *every* positive rational |
| $a, b$ | $\exists$ | We only need *some* coprime representation to exist |
| Connective | $\wedge$ (and) | Both $x = a/b$ and $G(a,b)$ must hold simultaneously |

### Why the wrong options fail

**$\exists a\; \exists b\; (x = a/b \to G(a,b))$:** An implication is vacuously true whenever its antecedent is false. Just pick any $a, b$ with $x \neq a/b$ -- the implication holds regardless of $G(a,b)$. This makes the statement trivially true for every $x$ without actually finding a coprime representation.

**$\exists a\; \exists b\; (G(a,b) \to x = a/b)$:** Same issue in the other direction. Pick $a = 1, b = 1$ (coprime). If $x \neq 1$, the implication is vacuously true without $x$ having any representation at all.

**$\forall a\; \forall b\; (x = a/b \wedge G(a,b))$:** Claims that *every* pair $(a, b)$ represents $x$ in lowest terms, which is false for all but trivial cases.

## Q12, Set Algebra Identity

Which set equals $A \cap \overline{(B \setminus C)}$?

**Answer:** $(A \cap \overline{B}) \cup (A \cap C)$

### Step-by-step derivation

**Step 1:** Rewrite set difference as intersection with complement:
$$B \setminus C = B \cap \overline{C}$$

**Step 2:** Apply De Morgan's law to the complement of that intersection:
$$\overline{B \cap \overline{C}} = \overline{B} \cup C$$

**Step 3:** Distribute $A$ across the union (distributive law):
$$A \cap (\overline{B} \cup C) = (A \cap \overline{B}) \cup (A \cap C)$$

### Why $A \cap \overline{B} \cap C$ is wrong

That equals $A \cap (C \setminus B)$, which keeps only elements of $A$ that are *in* $C$ and *not in* $B$.
The correct answer also includes elements of $A$ that are simply not in $B$, regardless of whether they are in $C$.

In [ ]:
U = set(range(10))
A = {0, 2, 4, 6, 8}
B = {1, 2, 3, 4, 5}
C = {3, 4, 5, 6, 7}
sets = {'A': A, 'B': B, 'C': C, 'U': U}

target_expr = "A inter ~(B without C)"

# Drop the option strings in from the MC question, the correct one will be marked
options = [
    "(A inter ~B) union (A inter C)",   # correct
    "A inter ~B inter C",
    "(A union B) inter (A union ~C)",
    "A inter ~B union C",
]

target = evaluate_set_expression(target_expr, sets, U)

print(f"Target  A ∩ ~(B\\C) = {sorted(target)}\n")

for opt in options:
    result = evaluate_set_expression(opt, sets, U)
    mark = '✓' if result == target else '✗'
    print(f"  {mark}  {opt}")
    print(f"       = {sorted(result)}")

## Q13, Empty Set Membership

The empty set $\emptyset$ is an element of which sets?

**Answer:** $\{\emptyset\}$

**Trick summary:**
- $\{\emptyset\}$, YES: this set contains exactly one element, which is $\emptyset$.
- $\emptyset$, NO: the empty set has no elements.
- $\{\{\emptyset\}\}$, NO: contains $\{\emptyset\}$, not $\emptyset$.
- $\{x \in \mathbb{R} : x < x\}$ = $\emptyset$, NO: the empty set has no elements.

In [ ]:
truths = empty_set_option_truths()
for name, val in truths.items():
    print(f"  ∅ ∈ {name}: {val}")

## Q14, Tautology Checker

Which is a tautology?

| Expression | Tautology? |
|---|---|
| $(p \to q) \vee (\neg q \to \neg p)$ | No |
| $p \leftrightarrow q$ | No |
| $(\neg p \vee \neg q) \to (\neg(p \wedge q))$ | **Yes** |
| $(p \vee q \vee r) \wedge (p \vee \neg q \vee \neg r)$ | No |

In [ ]:
# why is this a list of tuples?
expressions = [
    ("(p => q) or (not q => not p)"),
    ("p <=> q"),
    ("(not p or not q) => (not (p and q))"),
    ("(p or q or r) and (p or not q or not r)"),
    ("p <=> p")
]

for expr in expressions:
    r = tautology_checker(expr)
    print(f"{expr:35s}  tautology: {r['is_tautology']}")

## Q15, Circular Seating at Two Tables

$3n$ people, one table with $n$ seats, one with $2n$ seats.

| Equivalence | Answer |
|---|---|
| Same left **and** right neighbor (oriented) | $\dfrac{(3n)!}{2n^2}$ |
| Same neighbors, no L/R distinction (unoriented) | $\dfrac{(3n)!}{8n^2}$ |

### Derivation

**Part a: same left and right neighbor**

Circular arrangements where rotation matters give $(k-1)!$ distinct seatings for $k$ people.

1. Choose which $n$ people sit at the small table: $\binom{3n}{n}$
2. Arrange them around the small table (oriented circular): $(n-1)!$
3. Arrange the remaining $2n$ people around the large table (oriented circular): $(2n-1)!$

$$\binom{3n}{n}(n-1)!(2n-1)! = \frac{(3n)!}{n!\,(2n)!} \cdot (n-1)! \cdot (2n-1)! = \frac{(3n)!}{n \cdot 2n} = \frac{(3n)!}{2n^2}$$

**Part b: same neighbors, no L/R distinction**

Each unoriented circular arrangement of $k$ people corresponds to exactly 2 oriented ones (the seating and its mirror). Divide by 2 for each table:

$$\frac{(3n)!}{2n^2} \times \frac{1}{2} \times \frac{1}{2} = \frac{(3n)!}{8n^2}$$

In [ ]:
# 3n people; table 1 has n seats, table 2 has 2n seats
n = 3
people = 3 * n
table1 = n
table2 = 2 * n

print(f"Setup: {people} people, tables of {table1} and {table2} seats\n")

lr   = seating_count_two_tables(n, distinguish_left_right=True)
no_lr = seating_count_two_tables(n, distinguish_left_right=False)

print(f"Same when same L+R neighbor : {lr}")
print(f"Same when same neighbors    : {no_lr}")

# Check against the MC answer option formulas
import sympy as sp
options = {
    "(3n)!/(2n²)": sp.Rational(math.factorial(3*n), 2*n**2),
    "(3n)!/(8n²)": sp.Rational(math.factorial(3*n), 8*n**2),
    "(3n)!/(4n²)": sp.Rational(math.factorial(3*n), 4*n**2),
    "(3n)!/(2n) ": sp.Rational(math.factorial(3*n), 2*n),
}
print(f"\nOption check for n={n}:")
for label, val in options.items():
    lr_mark   = '✓' if val == lr    else ' '
    nolr_mark = '✓' if val == no_lr else ' '
    print(f"  {label} = {val}   L/R:{lr_mark}  no-L/R:{nolr_mark}")

## Q16, Function Classification (Multi-part)

| Function | Classification |
|---|---|
| $f:\mathbb{R}\to\mathbb{Z},\; f(x)=2\lfloor x/2 \rfloor$ | Well-defined but neither |
| $f:\mathbb{R}\to\{x\geq 0\},\; f(x)=\sqrt{x^2}$ | Surjective |
| $f:\mathbb{N}\to\mathbb{N},\; f(x)=2x+7$ | Injective |
| $f:\mathbb{N}\to\mathbb{N},\; f(x)=x-x^2$ | Not well-defined (negative for $x>1$) |
| $f:\mathbb{R}\to\mathbb{R},\; f(x)=x$ if $x\in\mathbb{Q}$, $-x$ if $x\notin\mathbb{Q}$ | Both (bijective) |

In [ ]:
import math, fractions

# TODO: Stop printning large seperators, and remove hardcoded values, the classify function should be able to check the surjectivity, injectivity and well-definedness of the function.

print("1. f: ℝ → ℤ,  f(x) = 2⌊x/2⌋")
# Range: only even integers → not surjective onto ℤ
# Not injective: f(0) = f(0.5) = 0
samples = [-3, -2, -1, 0, 0.5, 1, 1.5, 2, 3]
vals = [2 * math.floor(x / 2) for x in samples]
print(f"  sample outputs: {vals}")

print("2. f: ℝ → {x≥0},  f(x) = √(x²) = |x|")
print("  Surjective: every y≥0 is hit (f(y)=y)")
print("  Not injective: f(1) = f(-1) = 1")
print("  => SURJECTIVE but not injective")


print("3. f: ℕ → ℕ,  f(x) = 2x + 7")
r = classify_function_finite(range(0, 30), range(0, 100), lambda x: 2*x + 7)
print(f"  {r['classification']} (range is only odd integers ≥ 7)")


print("4. f: ℕ → ℕ,  f(x) = x − x²")
r2 = classify_function_finite(range(0, 10), range(0, 100), lambda x: x - x**2)
print(f"  {r2['classification']}, counterexample: x={r2['counterexample'][0]} → {r2['counterexample'][1]}")

print("5. f: ℝ → ℝ,  f(x) = x if x∈ℚ, else −x")
# Injective: if both rational f(x)=x≠y=f(y); if x∈ℚ,y∉ℚ then x≠-y (different rationality)
# Surjective: any z∈ℚ achieved by z; any z∉ℚ achieved by -z (also irrational)
print("  Injective + surjective => BOTH (bijective)")
# Spot-check on rationals:
dom = [fractions.Fraction(i) for i in range(-5, 6)]
r5 = classify_function_finite(dom, list(range(-5, 6)), lambda x: x)
print(f"  Rational domain sample: {r5['classification']}")

In [ ]:
# Linear descriptor parser for simple cases
descriptor = "f: NN -> NN, f(x) = 2x + 7"
r = classify_linear_function_descriptor(descriptor)
print(descriptor)
print(" ->", r['classification'])
print()

## Q17, Permutations of ABCDE with Constraints (Multi-part)

| Constraint | Answer |
|---|---|
| Contain none of AB, BC, CD | $64$ |
| Contain ACE as subsequence | None of these ($= 20$) |
| Contain precisely one of AB, CD | $36$ |

In [ ]:
perms = get_permutations('ABCDE')
print(f"Total permutations of ABCDE: {len(perms)}\n")

# Contiguous substring constraints
none_abc = count_permutations_with_constraints('ABCDE', must_not_contain_any=['AB', 'BC', 'CD'])
print(f"None of AB, BC, CD (contiguous):  {none_abc}")

one_of_ab_cd = count_permutations_with_constraints('ABCDE', exactly_one_of=['AB', 'CD'])
print(f"Precisely one of AB, CD:           {one_of_ab_cd}")

contains_abc = count_permutations_with_constraints('ABCDE', must_contain_all=['ABC'])
print(f"Contain ABC (contiguous):          {contains_abc}")

ab_or_cd = count_permutations_with_constraints('ABCDE', must_contain_any=['AB', 'CD'])
print(f"Contain AB or CD:                  {ab_or_cd}")

# Non-contiguous subsequence constraint
# Use count_permutations_subsequence: A appears before C, C before E (anywhere)
ace_count = count_permutations_subsequence('ABCDE', 'ACE')
print(f"\nACE as subsequence (A<C<E pos):    {ace_count}  <- 'None of these' (not in options 12,24,36,48,50,64)")

ab_subseq = count_permutations_subsequence('ABCDE', 'AB')
print(f"AB as subsequence (A before B):    {ab_subseq}")

# ── Template: swap in your own letters/patterns ──────────────────────────
# items      = 'ABCDE'                          # change the set
# patterns   = ['XY', 'YZ']                     # contiguous substrings to check
# subseq     = 'ACE'                            # non-contiguous subsequence
# count_permutations_with_constraints(items, must_not_contain_any=patterns)
# count_permutations_subsequence(items, subseq)

## Q18, Relation Classification on $\{a,b,c,d\}$ (Multi-part)

| Relation | Classification |
|---|---|
| $\{(a,a),(a,b),(a,c),(a,d)\}$ | None |
| $\{(a,b),(b,c),(c,d)\}$ | Hasse / covering relation |
| $\{(a,a),(b,b),(c,c),(d,d),(a,d),(d,a)\}$ | Equivalence relation |
| $\{(a,a),(b,b),(c,c),(d,d),(d,c)\}$ | Partial order |
| $\{(a,a),(b,b),(c,c),(d,d),(a,b),(a,c),(a,d),(b,c),(b,d),(c,d)\}$ | Well order |

In [ ]:
S = {'a', 'b', 'c', 'd'}

relations = [
    {('a','a'),('a','b'),('a','c'),('a','d')},
    {('a','b'),('b','c'),('c','d')},
    {('a','a'),('b','b'),('c','c'),('d','d'),('a','d'),('d','a')},
    {('a','a'),('b','b'),('c','c'),('d','d'),('d','c')},
    {('a','a'),('b','b'),('c','c'),('d','d'),
     ('a','b'),('a','c'),('a','d'),('b','c'),('b','d'),('c','d')},
]

for i, R in enumerate(relations, 1):
    c = classify_relation(S, R)
    props = [k for k in ['reflexive','irreflexive','symmetric','antisymmetric','transitive'] if c[k]]
    print(f"R{i}: {c['classification']}")
    print(f"     props: {props}\n")

## Q19, Recursive Tiling of an $n \times 2$ Board

Tiles are $2 \times 1$. The board has $n$ columns and 2 rows.

**Answer:** $f(0) = 1,\; f(1) = 1,\; f(n) = f(n-1) + f(n-2)$ for $n \geq 2$

This is the Fibonacci sequence (shifted so both base cases are 1).

### Why the recurrence holds

Consider how the **rightmost column** is covered -- there are exactly two cases:

- **Vertical tile:** one $2 \times 1$ tile placed vertically fills column $n$ completely. The remaining board is $(n-1) \times 2$, giving $f(n-1)$ tilings.
- **Two horizontal tiles:** one horizontal tile on each row must cover both columns $n-1$ and $n$ together. The remaining board is $(n-2) \times 2$, giving $f(n-2)$ tilings.

These cases are mutually exclusive and exhaustive, so $f(n) = f(n-1) + f(n-2)$.

### Why the base cases are 1, not 0

- **$f(0) = 1$:** the empty board has exactly one tiling -- the empty one. This is not zero.
  Concretely: if $f(0) = 0$, then $f(2) = f(1) + f(0) = 1 + 0 = 1$, but a $2 \times 2$ board has 2 tilings (both columns vertical, or both rows horizontal). So $f(0)$ must be 1.
- **$f(1) = 1$:** a $1 \times 2$ board is covered by exactly one vertical tile.

### Why $f(n) = 2f(n-2)$ is wrong

That recurrence counts only tilings where every pair of rightmost columns is covered by two horizontal tiles. It ignores all tilings that end with a vertical tile in the last column.

In [ ]:
# Brute-force: tile n×2 board with 2×1 tiles
# Fact: exactly the Fibonacci numbers (f(n)=f(n-1)+f(n-2), f(0)=f(1)=1)
def brute_force_tiling(n, memo={}):
    if n in memo: return memo[n]
    if n <= 1: return 1
    memo[n] = brute_force_tiling(n-1) + brute_force_tiling(n-2)
    return memo[n]

brute = [brute_force_tiling(n) for n in range(9)]
print(f"Brute-force f(0..8): {brute}\n")

# Test each candidate recurrence from the MC options
def test_recurrence(name, base_cases, rule, n_max=8):
    memo = dict(base_cases)
    def f(n):
        if n not in memo: memo[n] = rule(n, f)
        return memo[n]
    candidate = [f(n) for n in range(n_max + 1)]
    ok = candidate == brute[:n_max + 1]
    print(f"  {'✓ CORRECT' if ok else '✗ wrong  '} {name}")
    print(f"            {candidate}")

candidates = [
    ("f(0)=1, f(1)=1, f(n)=f(n-1)+f(n-2)", {0:1, 1:1}, lambda n,f: f(n-1)+f(n-2)),
    ("f(0)=0, f(1)=1, f(n)=f(n-1)+f(n-2)", {0:0, 1:1}, lambda n,f: f(n-1)+f(n-2)),
    ("f(0)=1, f(1)=1, f(n)=2·f(n-2)",       {0:1, 1:1}, lambda n,f: 2*f(n-2)),
    ("f(0)=0, f(1)=1, f(n)=2·f(n-2)",       {0:0, 1:1}, lambda n,f: 2*f(n-2)),
    ("f(0)=1, f(1)=1, f(n)=f(n-1)·f(n-2)", {0:1, 1:1}, lambda n,f: f(n-1)*f(n-2)),
]
for name, base, rule in candidates:
    test_recurrence(name, base, rule)

## Q20, Coefficient of $x^{15}y^{20}$ (Multi-part)

| Polynomial | Coefficient of $x^{15}y^{20}$ |
|---|---|
| $(2x^3 - y^4)^{10}$ | $-\binom{10}{5} \cdot 2^5$ |
| $(1 - 2x^3 y^4)^{10}$ | $-\binom{10}{5} \cdot 2^5$ |
| $(x^3 - 2y^4)^{10}$ | $-\binom{10}{5} \cdot 2^5$ |

In [ ]:
from math import comb as _comb

# TODO: use LaTeX formatting so output is readable.
def format_coeff(coeff):
    """Try to express an integer as ±C(n,k)·b^e for small n,k,b,e."""
    c = int(coeff)
    sign, c_abs = (-1, -c) if c < 0 else (1, c)
    best = str(coeff)  # fallback
    for b in range(2, 12):
        for e in range(0, 25):
            bpow = b ** e
            if bpow > c_abs: break
            if c_abs % bpow == 0:
                rem = c_abs // bpow
                for nn in range(0, 20):
                    for kk in range(0, nn + 1):
                        if _comb(nn, kk) == rem:
                            s = '-' if sign < 0 else ''
                            best = f"{s}C({nn},{kk})·{b}^{e}" if e > 0 else f"{s}C({nn},{kk})"
                            return best  # first (smallest) match
    return best

target_powers = {'x': 15, 'y': 20}

polys = {
    "(2x³ - y⁴)¹⁰": expand((2*x**3 - y**4)**10),
    "(1 - 2x³y⁴)¹⁰": expand((1 - 2*x**3*y**4)**10),
    "(x³ - 2y⁴)¹⁰":  expand((x**3 - 2*y**4)**10),
}

print(f"Looking for coefficient of x^15 · y^20\n")
for label, poly in polys.items():
    coeff = coefficient_of_monomial(poly, target_powers)
    print(f"  {label}: {coeff}  =  {format_coeff(coeff)}")

## Q21, Logical Equivalences for "$a$ and $b$ are relatively prime"

**Answer:** All three are equivalent.

The three statements, where the domain is all positive integers:

1. $\neg(\exists c\;(c \mid a \wedge c \mid b \wedge c > 1))$
2. $\forall c\;(\neg(c \mid a) \vee \neg(c \mid b) \vee c \leq 1)$
3. $\forall c\;((c \mid a \wedge c \mid b) \implies c \leq 1)$

### Transformation chain

**Statement 1 to Statement 2** -- push the negation inward using De Morgan's laws:

$$\neg\exists c\; P(c) \;\equiv\; \forall c\; \neg P(c)$$

$$\neg(c \mid a \;\wedge\; c \mid b \;\wedge\; c > 1) \;\equiv\; \neg(c \mid a) \;\vee\; \neg(c \mid b) \;\vee\; \neg(c > 1) \;\equiv\; \neg(c \mid a) \;\vee\; \neg(c \mid b) \;\vee\; c \leq 1$$

**Statement 2 to Statement 3** -- use $P \to Q \;\equiv\; \neg P \vee Q$, reading right to left:

$$\neg(c \mid a) \vee \neg(c \mid b) \vee c \leq 1 \;\equiv\; \neg(c \mid a \wedge c \mid b) \vee c \leq 1 \;\equiv\; (c \mid a \wedge c \mid b) \to c \leq 1$$

All three say the same thing: no integer greater than 1 divides both $a$ and $b$.

In [ ]:
def check_coprimality_equiv(stmt_func, name, test_range=30):
    """Verify stmt_func(a,b) agrees with gcd(a,b)==1 for all pairs in [1, test_range]."""
    for a in range(1, test_range + 1):
        for b in range(1, test_range + 1):
            expected = math.gcd(a, b) == 1
            if stmt_func(a, b) != expected:
                print(f"  ✗ {name}  FAILS at ({a},{b}): got {stmt_func(a,b)}, expected {expected}")
                return
    print(f"  ✓ {name}")

# The three statements from Q21
stmt1 = lambda a,b: not any(a%c==0 and b%c==0 and c>1 for c in range(2, min(a,b)+1))
stmt2 = lambda a,b: all(a%c!=0 or b%c!=0 or c<=1 for c in range(1, min(a,b)+1))
stmt3 = lambda a,b: all(not(a%c==0 and b%c==0) or c<=1 for c in range(1, min(a,b)+1))


print("Checking against gcd(a,b)==1 for all pairs in [1,30]²:\n")
check_coprimality_equiv(stmt1, "~∃c(c|a ∧ c|b ∧ c>1)")
check_coprimality_equiv(stmt2, "∀c(~(c|a) ∨ ~(c|b) ∨ c≤1)")
check_coprimality_equiv(stmt3, "∀c((c|a ∧ c|b) → c≤1)")

# Template: add your own statement here
# my_stmt = lambda a, b: ...  <your predicate logic here>
# check_coprimality_equiv(my_stmt, 'description')

# E24 Exam

## Q1 — Divisibility

If $a, b, c, d$ are positive integers such that $ab \mid cd$, which of the following **must** be true?

### How to approach “must be true” questions

**Must be true** = holds for **every** valid $(a,b,c,d)$ where $ab \mid cd$.
Strategy: try to find a **counterexample** for each option. One counterexample kills it.
The surviving option is the answer — then ask *why* it always holds.

---

### Options and their fate

| Option | Verdict | Counterexample (if any) |
|---|---|---|
| $a \mid \text{lcm}(c,d)$ or $b \mid \text{lcm}(c,d)$ | **False** | $a=4,b=9,c=6,d=6$: $\text{lcm}=6$, neither $4$ nor $9$ divides $6$ |
| **If $p$ prime divides $a$, then $p \mid c$ or $p \mid d$** | **True** | See proof below |
| None of these | **False** | (option 2 is true) |
| If $b$ prime, then $b \mid \gcd(c,d)$ | **False** | $a=1,b=2,c=4,d=1$: $ab=2\mid 4=cd$, but $\gcd(4,1)=1$ and $2\nmid 1$ |
| $\gcd(a,b) \mid \gcd(c,d)$ | **False** | $a=6,b=10,c=15,d=4$: $ab=60\mid 60=cd$, $\gcd(6,10)=2$, $\gcd(15,4)=1$, $2\nmid 1$ |
| $a\mid c$ or $a\mid d$ or $b\mid c$ or $b\mid d$ | **False** | $a=4,b=9,c=6,d=6$: none of the four divide $6$ |

---

### Why option 2 is always true (Euclid’s lemma)

**Given:** $ab \mid cd$ and $p \mid a$ where $p$ is prime.

**Chain:** $p \mid a$ and $a \mid ab$, so $p \mid ab$. Since $ab \mid cd$, we get $p \mid cd$.

**Euclid’s lemma:** if a prime $p$ divides a product $cd$, then $p \mid c$ **or** $p \mid d$. Done.

> Note: this only works because $p$ is **prime**. For composite divisors the lemma breaks — that’s why option 4 (non-prime $b$) and option 6 (just $a$ or $b$, not a prime factor) both fail.

In [ ]:
from math import gcd
from sympy import isprime, lcm as sym_lcm

# Brute-force: find first counterexample for each option over small integers
LIMIT = 15

def find_counterexample(predicate):
    for a in range(1, LIMIT):
        for b in range(1, LIMIT):
            for c in range(1, LIMIT):
                for d in range(1, LIMIT):
                    if (c * d) % (a * b) == 0:  # ab | cd
                        if not predicate(a, b, c, d):
                            return (a, b, c, d)
    return None

options = [
    ('1. a|lcm(c,d) OR b|lcm(c,d)',
        lambda a,b,c,d: sym_lcm(c,d) % a == 0 or sym_lcm(c,d) % b == 0),

    ('2. p prime, p|a => p|c or p|d',
        lambda a,b,c,d: all(
            (c % p == 0 or d % p == 0)
            for p in range(2, a+1)
            if isprime(p) and a % p == 0
        )),

    ('4. b prime => b|gcd(c,d)',
        lambda a,b,c,d: not isprime(b) or gcd(c,d) % b == 0),

    ('5. gcd(a,b) | gcd(c,d)',
        lambda a,b,c,d: gcd(c,d) % gcd(a,b) == 0),

    ('6. a|c or a|d or b|c or b|d',
        lambda a,b,c,d: c%a==0 or d%a==0 or c%b==0 or d%b==0),
]

for label, pred in options:
    cex = find_counterexample(pred)
    if cex:
        a, b, c, d = cex
        print(f'FAIL  {label}')
        print(f'      counterexample: a={a}, b={b}, c={c}, d={d}  (ab={a*b}, cd={c*d})')
    else:
        print(f'PASS  {label}  -- no counterexample up to {LIMIT}')


FAIL  1. a|lcm(c,d) OR b|lcm(c,d)
      counterexample: a=4, b=9, c=6, d=6  (ab=36, cd=36)
PASS  2. p prime, p|a => p|c or p|d  -- no counterexample up to 15
FAIL  4. b prime => b|gcd(c,d)
      counterexample: a=1, b=2, c=1, d=2  (ab=2, cd=2)
FAIL  5. gcd(a,b) | gcd(c,d)
      counterexample: a=2, b=2, c=1, d=4  (ab=4, cd=4)
FAIL  6. a|c or a|d or b|c or b|d
      counterexample: a=4, b=9, c=6, d=6  (ab=36, cd=36)


the correct answer is: **If $p$ is a prime that divides $a$, then $p \mid c$ or $p \mid d$** because of Euclid's lemma: if a prime divides a product, it must divide at least one factor. Since $ab \mid cd$, we have $cd = k \cdot ab$ for some integer $k$. If a prime $p$ divides $a$, then $p$ divides $ab$, and thus $p$ must divide $cd$. By Euclid's lemma, $p$ must divide either $c$ or $d$.

## Q2 — N-cube Edges

For $n \geq 4$, the number of edges in the $n$-cube $Q_n$ is:

- None of these
- $2^n + 16$
- $n \cdot 2^{n-1}$
- $2^{n+1}$
- $n! + 8$

In [ ]:
# Check with hypercube_info
n = 4
hypercube_info(n)

# extract the edge number and check against options:

edges = hypercube_info(n)['edges']
print(f"Number of edges in {n}-cube: {edges}")

options = {
    "n*2^(n-1)": n * 2**(n-1),
    "2^n": 2**n,
    "n^2": n**2,
    "n*2^n": n * 2**n,
}
print("\nOption check:")
for label, val in options.items():
    mark = '✓' if val == edges else '✗'
    print(f"  {mark} {label} = {val}")

Number of edges in 4-cube: 32

Option check:
  ✓ n*2^(n-1) = 32
  ✗ 2^n = 16
  ✗ n^2 = 16
  ✗ n*2^n = 64


## Q3 — Function Classification (Multi-part)

For each function, pick one: **Not well-defined / Neither / Surjective only / Injective only / Both**

Recall $\mathbb{N}$ = nonneg integers, $\mathbb{Z}^+$ = positive integers.

1. $f : \mathbb{Z}^+ \to \mathbb{N}$, $\quad f(x) = \lfloor \log_2(x) \rfloor$
2. $f : \mathbb{R} \to \mathbb{Z}^+$, $\quad f(x) = \lfloor |x| \rfloor$
3. $f : \mathbb{N} \to \mathbb{Z}$, $\quad f(x) = \lceil x/2 \rceil$ if $x$ even, $-\lceil x/2 \rceil$ if $x$ odd
4. $f : \mathbb{R} \to \{-1, 0, 1\}$, $\quad f(x) = \lfloor x \rfloor - \lceil x \rceil$
5. $f : \mathbb{N} \to \mathbb{N}$, $\quad f(x) = x^3 + 1$

In [ ]:
import math

# NOTE: surjectivity check requires codomain = exact set of values f *should* hit.
# If you pass range(0,100) but f only reaches 0-5, it'll wrongly say "not surjective".

functions = [
    ("f1: Z+->N,  floor(log2(x))",    range(1, 33),        range(0, 6),    lambda x: int(math.log2(x))),
    ("f2: R->Z+,  floor(|x|)",         [-1,-0.5,0,0.5,1,2], range(1, 10),   lambda x: int(abs(x))),
    ("f3: N->Z,   piecewise ceil",     range(0, 20),        range(-10, 10), lambda x: math.ceil(x/2) if x%2==0 else -math.ceil(x/2)),
    ("f4: R->{-1,0,1}, floor-ceil",   [-1,0,1,0.5,1.5],    [-1, 0, 1],     lambda x: math.floor(x) - math.ceil(x)),
    ("f5: N->N,   x^3+1",              range(0, 10),        range(0, 731),  lambda x: x**3 + 1),
]

for label, dom, cod, func in functions:
    r = classify_function_finite(dom, cod, func)
    print(f"{label:35s}  {r['classification']}")
    if not r['well_defined']:
        print(f"  counterexample: f({r['counterexample'][0]}) = {r['counterexample'][1]}")


f1: Z+->N,  floor(log2(x))           Surjective but not injective
f2: R->Z+,  floor(|x|)               not well defined
  counterexample: f(-0.5) = 0
f3: N->Z,   piecewise ceil           Both surjective and injective
f4: R->{-1,0,1}, floor-ceil          Well defined but neither surjective nor injective
f5: N->N,   x^3+1                    Injective but not surjective


## Q4 — Complete Graph $K_{2n}$ Edges

Consider these three statements about $K_{2n}$:
- **(a):** $K_{2n}$ has $\binom{2n}{2}$ edges
- **(b):** $K_{2n}$ has $2\binom{n}{2} + n^2$ edges
- **(c):** $K_{2n}$ has $n(2n-1)$ edges

Which are true?

- (a) and (b) true, (c) false
- All three true
- (a) and (c) true, (b) false
- (b) and (c) true, (a) false
- Precisely one is true

In [ ]:
from math import comb

for n in range(1, 8):
    actual = complete_graph_edges(2 * n)
    a = comb(2*n, 2)
    b = 2*comb(n, 2) + n**2
    c = n * (2*n - 1)
    print(f"n={n}:  K_2n has {actual} edges  |  (a)={a}  (b)={b}  (c)={c}")


n=1:  K_2n has 1 edges  |  (a)=1  (b)=1  (c)=1
n=2:  K_2n has 6 edges  |  (a)=6  (b)=6  (c)=6
n=3:  K_2n has 15 edges  |  (a)=15  (b)=15  (c)=15
n=4:  K_2n has 28 edges  |  (a)=28  (b)=28  (c)=28
n=5:  K_2n has 45 edges  |  (a)=45  (b)=45  (c)=45
n=6:  K_2n has 66 edges  |  (a)=66  (b)=66  (c)=66
n=7:  K_2n has 91 edges  |  (a)=91  (b)=91  (c)=91


Since all of a,b,c seem to have the same values for multiple $n$ values, the answer is likely **All three true**.

## Q5 — Degree Sequence Existence

Consider a simple graph with degrees $2, 2, 3, 3, 3, 3, 3$.

- Such a graph exists, and any such graph has fewer than 19 edges
- Such a graph does not exist
- Such a graph exists, and any such graph has precisely 19 edges
- Such a graph exists, and any such graph has more than 19 edges
- There exist such graphs with more than 19 edges and others with fewer than 19 edges

In [ ]:
degrees = [2, 2, 3, 3, 3, 3, 3]

# sum degrees
total_degrees = sum(degrees)
print(f"Total degrees: {total_degrees}")


Total degrees: 19


By the handshake lemma, if the sum of degrees must be even for a graph to exist. since the sum here is $2+2+3+3+3+3+3=19$, which is odd, such a graph cannot exist. Therefore, the correct answer is: **Such a graph does not exist**.

## Q6 — Inclusion-Exclusion (4 Sets)

How many elements are in the union of four sets if:
- Each set has 200 elements
- Each pair of sets shares 50 elements
- Each triple of sets shares 25 elements
- All four sets share 5 elements

- 395
- 695
- 495
- 595
- None of these

In [ ]:
# inclusion_exclusion_uniform(n_sets, [each_single, each_pair, each_triple, ...])
print(inclusion_exclusion_uniform(4, [200, 50, 25, 5]))


595


Due to the symmetry of the problem, we can apply the inclusion-exclusion principle directly, using the function, we can calculate the number:

$$4 \cdot 200 - \binom{4}{2} \cdot 50 + \binom{4}{3} \cdot 25 - \binom{4}{4} \cdot 5 = 800 - 300 + 100 - 5 = 595$$


## Q7 — Cancellation in Modular Arithmetic

Suppose $a, b, c, m$ are positive integers with $ac \equiv bc \pmod{m}$. Which **must** be true?

- $a \equiv b \pmod{m}$
- $a - b \in \{km : k \in \mathbb{Z}\}$ if $\gcd(c, m) = 1$
- $c \mid m(a-b)$
- $a \equiv b \pmod{cm}$
- None of these

In [ ]:
from math import gcd
# we know the valid expression is:
def valid(a, b, c, m):
    return (a*c - b*c) % m == 0

# we bruteforce check for counterexamples to each option:
def find_counterexample(predicate, limit=15):
    for a in range(1, limit):
        for b in range(1, limit):
            for c in range(1, limit):
                for m in range(1, limit):
                    if valid(a, b, c, m) and not predicate(a, b, c, m):
                        return (a, b, c, m)
    return None
# yes this is painfully slow, but it works for small limits.

options = [
    ("opt1: a≡b (mod m)",            lambda a,b,c,m: (a-b) % m == 0),
    ("opt2: gcd(c,m)=1 => m|(a-b)",  lambda a,b,c,m: gcd(c,m) != 1 or (a-b) % m == 0),
    ("opt3: c | m(a-b)",             lambda a,b,c,m: (m*(a-b)) % c == 0),
    ("opt4: a≡b (mod cm)",           lambda a,b,c,m: (a-b) % (c*m) == 0),
]


for label, pred in options:
    cex = find_counterexample(pred)
    if cex:
        a,b,c,m = cex
        print(f"FAIL {label}  cex: a={a},b={b},c={c},m={m}")
    else:
        print(f"PASS {label}")

FAIL opt1: a≡b (mod m)  cex: a=1,b=2,c=2,m=2
PASS opt2: gcd(c,m)=1 => m|(a-b)
FAIL opt3: c | m(a-b)  cex: a=1,b=2,c=2,m=1
FAIL opt4: a≡b (mod cm)  cex: a=1,b=2,c=2,m=1


## Q8 — What Does NOT Necessarily Hold

If $2a + 3 \equiv 2b + 3 \pmod{m}$ for positive integers $a, b, m$, which does **NOT** necessarily have to be true?

- $2a \equiv 2b \pmod{m}$
- $a = km + b$ for some $k \in \mathbb{Z}$
- $14a \equiv 14b + 7m \pmod{m}$
- All of these are true
- $m \mid 2(a-b)$

In [ ]:
from math import gcd
# we know the valid expression is:
def valid(a, b, m):
    return (2*a + 3 - (2*b + 3)) % m == 0

# we bruteforce check for counterexamples to each option:
def find_counterexample(predicate, limit=15):
    for a in range(1, limit):
        for b in range(1, limit):
            for c in range(1, limit):
                for m in range(1, limit):
                    if valid(a, b, m) and not predicate(a, b, m):
                        return (a, b, m)
    return None
# yes this is painfully slow, but it works for small limits.

options = [
    ("opt1: a≡b (mod m)",            lambda a,b,m: (2*a - 2*b) % m == 0),
    ("opt2: gcd(2,m)=1 => m|(a-b)",  lambda a,b,m: (a - b) % m == 0),
    ("opt3: 2 | m(a-b)",             lambda a,b,m: (14*a - 14*b - 7*m) % m == 0),
    ("opt4: a≡b (mod 2m)",           lambda a,b,m: (2*a - 2*b) % m == 0),
]

for label, pred in options:
    cex = find_counterexample(pred)
    if cex:
        a,b,m = cex
        print(f"FAIL {label}  cex: a={a},b={b},m={m}")
    else:
        print(f"PASS {label}")



PASS opt1: a≡b (mod m)
FAIL opt2: gcd(2,m)=1 => m|(a-b)  cex: a=1,b=2,m=2
PASS opt3: 2 | m(a-b)
PASS opt4: a≡b (mod 2m)


since the code fails at one point, the answer must be that point; $a = k m$ for some ZZ 

## Q9 — Partitions of $\mathbb{Z} \times \mathbb{Z}$

Which of these are partitions of $\mathbb{Z} \times \mathbb{Z}$?
- **(a):** pairs where $x$ or $y$ (or both) are odd; pairs where $x$ and $y$ are even
- **(b):** pairs where $x$ and $y$ are odd; pairs where $x$ and $y$ are even
- **(c):** pairs where $x$ or $y$ (or both) are odd; pairs where $x$ or $y$ (or both) are even

Options:
- None of them
- Only (a)
- Precisely two of them
- Only (b)
- Only (c)

In [ ]:
# Check: do the parts cover all pairs? Are they disjoint?
# Try (0,0), (1,0), (1,1) etc.

# represent each part as a function: (x,y) -> bool
candidates = {
    'a': [
        lambda x,y: x%2==1 or y%2==1,   # x or y odd
        lambda x,y: x%2==0 and y%2==0,  # x and y even
    ],
    'b': [
        lambda x,y: x%2==1 and y%2==1,  # x and y odd
        lambda x,y: x%2==0 and y%2==0,  # x and y even
    ],
    'c': [
        lambda x,y: x%2==1 or y%2==1,   # x or y odd
        lambda x,y: x%2==0 or y%2==0,   # x or y even
    ],
}

sample = range(-5, 6)
for name, (p1, p2) in candidates.items():
    gaps    = [(x,y) for x in sample for y in sample if not p1(x,y) and not p2(x,y)]
    overlaps = [(x,y) for x in sample for y in sample if p1(x,y) and p2(x,y)]
    print(f"({name}): gaps={len(gaps)}  overlaps={len(overlaps)}  {'PARTITION' if not gaps and not overlaps else 'NOT a partition'}")




(a): gaps=0  overlaps=0  PARTITION
(b): gaps=60  overlaps=0  NOT a partition
(c): gaps=0  overlaps=60  NOT a partition


since only $a$ satisfies no gaps and no overlaps, the answer is: **Only (a)**.

## Q10 — Simple Graphs with All Degrees Distinct

Statement: *There exist infinitely many simple graphs such that all degrees are distinct.*

This is:
- Neither true nor false (depends on the graph)
- False by the pigeonhole principle
- False by counterexample
- True by the pigeonhole principle
- True by induction

The minimum edges of a simple graph is $n-1$ then the pigeon hole principle applies that there must be two vertices with the same degree. Therefore, the answer is: **False by the pigeonhole principle**. 

## Q11 — Binomial Theorem Coefficients (Multi-part)

Consider $(2x^2 - 3y^3)^8$.

| Sub-question | $0$ | $2^4 3^4 \binom{8}{4}$ | $-2^4 3^4 \binom{8}{4}$ | $2^3 3^6 \binom{8}{4}$ | $-2^4 3^6 \binom{8}{4}$ |
|---|---|---|---|---|---|
| Coefficient of $x^8 y^{12}$ | | | | | |
| Coefficient of $x^6 y^9$ | | | | | |

In [ ]:
from sympy import symbols, expand
x, y = symbols('x y')
poly = expand((2*x**2 - 3*y**3)**8)
# coefficient_of_monomial(poly, {'x': ?, 'y': ?})

coefficient_of_monomial(poly, {'x': 8, 'y': 12})

# we check against the possible options:
print(f"Coefficient of x^8 y^12: {coefficient_of_monomial(poly, {'x': 8, 'y': 12}) == 2**4 * 3**4 * comb(8,4)}")

print(f"Coefficient of x^6 y^9 = 0?: {coefficient_of_monomial(poly, {'x': 6, 'y': 9}) == 0}")



Coefficient of x^8 y^12: True
Coefficient of x^6 y^9 = 0?: True


## Q12 — Polynomial Divisibility

The polynomial $x^n + 1$ is divisible by $x + 1$ (for positive integer $n$):

- For each $n \geq 1$
- For each odd $n \geq 1$, and for no even $n$
- For each even $n \geq 4$, and for no odd $n$
- For infinitely many but not all even $n$, and for no odd $n$
- For $n = 1$ only

In [ ]:
from sympy import Poly, symbols, factor
x = symbols('x')
# Test: when does (x^n + 1) % (x + 1) == 0 ?
# Hint: substitute x = -1

x = -1

for n in range(1, 10):
    val = (x)**n + 1
    print(f'n={n}: (-1)^n + 1 = {val}, divisible: {val == 0}')


n=1: (-1)^n + 1 = 0, divisible: True
n=2: (-1)^n + 1 = 2, divisible: False
n=3: (-1)^n + 1 = 0, divisible: True
n=4: (-1)^n + 1 = 2, divisible: False
n=5: (-1)^n + 1 = 0, divisible: True
n=6: (-1)^n + 1 = 2, divisible: False
n=7: (-1)^n + 1 = 0, divisible: True
n=8: (-1)^n + 1 = 2, divisible: False
n=9: (-1)^n + 1 = 0, divisible: True


It's true for every n >= 1 for every odd n.

## Q13 — Minimum Cables (Hall's Theorem)

The least number of cables to connect **10 computers** to **5 printers** such that for every choice of 5 of the 10 computers, those 5 can directly access 5 different printers.

- 40
- None of these
- $\binom{10}{5}$
- 30
- 50

In [ ]:

# Think: Hall's marriage theorem. Each set of 5 computers needs 5 distinct printers.
# What minimum degree (cables per computer) guarantees this?

# 10 computers, to 5 printer, so each computer has 5 printers
halls_min_degree(10, 5,5)


30

## Q14 — Relation Classification (Multi-part)

Relations on $\{1,2,3,4,5,6\}$:
- $R_1 = \{(1,2),(2,3),(1,3),(4,5),(5,6),(4,6)\}$
- $R_2 = \{(1,1),(2,2),(3,3),(4,4),(5,5),(6,6),(1,2),(3,4),(5,6),(1,6)\}$
- $R_3 = \{(1,2),(2,3),(3,4),(4,5),(5,6),(1,3),(2,4),(3,5),(4,6)\}$

| Relation | Equivalence | Partial order | Trans+Refl, not antisym | Trans+Antisym, not refl | None |
|---|---|---|---|---|---|
| $R_3$ | | | | | |
| $R_2$ | | | | | |
| $R_1$ | | | | | |

In [ ]:
S = {1, 2, 3, 4, 5, 6}
R1 = {(1,2),(2,3),(1,3),(4,5),(5,6),(4,6)}
R2 = {(1,1),(2,2),(3,3),(4,4),(5,5),(6,6),(1,2),(3,4),(5,6),(1,6)}
R3 = {(1,2),(2,3),(3,4),(4,5),(5,6),(1,3),(2,4),(3,5),(4,6)}
# classify_relation(S, R1)

classify_relation(S, R1)

classify_relation(S, R2)

classify_relation(S, R3)



{'reflexive': False,
 'irreflexive': True,
 'symmetric': False,
 'antisymmetric': True,
 'transitive': False,
 'classification': ['Hasse / covering relation']}

## Q15 — Multiplicative Inverse mod 9 (Multi-part)

For each $n$, find the multiplicative inverse of $n \bmod 9$, or state it does not exist.

| | Does not exist | 0 | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|---|---|
| $n = 6$ | | | | | | | | |
| $n = 2$ | | | | | | | | |
| $n = 7$ | | | | | | | | |

In [ ]:
import math
for n in [6, 2, 7]:
    if math.gcd(n, 9) == 1:
        inv = pow(n, -1, 9)
        print(f'{n}^(-1) mod 9 = {inv}')
    else:
        print(f'{n} has no inverse mod 9 (gcd={math.gcd(n,9)})')




6 has no inverse mod 9 (gcd=3)
2^(-1) mod 9 = 5
7^(-1) mod 9 = 4


## Q16 — Set Identity

Given universal set $U$, which equals $(\overline{B} - A) \cup (\overline{C} - A)$?

- $\emptyset$
- $\overline{(B \cup C)} - A$
- $((U - B) \cup (U - C)) \cap A$
- $\overline{(B \cap C)} - A$
- None of these

In [ ]:
U = set(range(10))
sets = {'A': {0,1,2,3}, 'B': {1,2,4,5}, 'C': {2,3,5,6}, 'U': U}
target = evaluate_set_expression('(~B without A) union (~C without A)', sets, U)
print('Target:', sorted(target))
# Test each option

options = [
    "~(B union C) without A",
    "((U without B) union (U without C)) inter A",
    "~(B inter C) without A",
]

for opt in options:
    result = evaluate_set_expression(opt, sets, U)
    mark = '✓' if result == target else '✗'
    print(f"  {mark}  {opt}")
    print(f"       = {sorted(result)}")


Target: [4, 6, 7, 8, 9]
  ✗  ~(B union C) without A
       = [7, 8, 9]
  ✗  ((U without B) union (U without C)) inter A
       = [0, 1, 3]
  ✓  ~(B inter C) without A
       = [4, 6, 7, 8, 9]


## Q17 — Circular Seating (Multi-part)

20 people seated around a round table.

| Scenario | $2^{20}$ | $2^{19}$ | $20!$ | $19!$ | $19!/2$ |
|---|---|---|---|---|---|
| Same if each person has same two neighbors (no L/R distinction) | | | | | |
| Same if each person has same left and right neighbor | | | | | |
| Same if each person has same left neighbor | | | | | |

TypeError: circular_arrangements() got an unexpected keyword argument 'distinguish_left_right'

## Q18 — Bézout's Identity

Let $a$ and $b$ be positive integers. Which of the following **CANNOT** necessarily be written as $as + bt$ for some integers $s, t$?

- All of these can be written as $as + bt$
- $\gcd(a,b)^2 - 17\,\text{lcm}(a,b)$
- $\text{lcm}(a,b) / \gcd(a,b)$
- $\gcd(2a, 6b)$
- $\gcd(a,b)$

In [ ]:
import math
# By Bezout: as+bt can represent exactly the multiples of gcd(a,b)
# So check: is each option always a multiple of gcd(a,b)?
# Try a=6, b=10: gcd=2, lcm=30
a, b = 6, 10
g = math.gcd(a, b)
l = (a * b) // g
print(f'a={a}, b={b}, gcd={g}, lcm={l}')
for label, val in [
    ('gcd^2 - 17*lcm', g**2 - 17*l),
    ('lcm/gcd', l // g),
    ('gcd(2a,6b)', math.gcd(2*a, 6*b)),
    ('gcd(a,b)', g),
]:
    print(f'  {label} = {val}, divisible by gcd? {val % g == 0}')


## Q19 — Recursive Definition of $\binom{n}{3}$

Which gives a recursive definition of the number of ways to choose 3 elements from an $n$-element set for $n \geq 3$?

- $f(3) = 1$ and $f(n+1) = \binom{n}{2} f(n-1)$ for $n \geq 3$
- None of these
- $f(4) = 1$ and $f(n) = n^4 + f(n-1)$ for $n > 4$
- $f(3) = 1$ and $f(n) = \binom{n}{3} + f(n-1)$ for $n > 3$
- $f(n) = n^3$
- $f(3) = 1$ and $f(n) = \frac{(n-1)(n-2)}{2} + f(n-1)$ for $n > 3$

In [ ]:
from math import comb, factorial
# Target: f(n) = C(n, 3)
for n in range(3, 10):
    print(f'C({n},3) = {comb(n, 3)}')
# Test candidate recurrences


## Q20 — Vandermonde's Identity

For integers $n, m$ with $2 \leq n \leq m$, the binomial coefficient $\binom{n+m}{m+2}$ equals:

- $\sum_{k=0}^{n} \binom{n}{k} \binom{m}{m-k}$
- $\sum_{k=2}^{n} \binom{n}{k} \binom{m}{m+2-k}$
- $\sum_{k=0}^{n+2} \binom{n}{k} \binom{m}{n-k}$
- None of these because we cannot use Vandermonde's identity in this case
- $\sum_{k=1}^{n} \binom{n}{k} \binom{m}{m+1-k}$

In [ ]:
from math import comb
# Vandermonde: C(m+n, r) = sum_{k} C(m,k)*C(n,r-k)
# Test each option for small n, m
n, m = 3, 5
target = comb(n + m, m + 2)
print(f'C({n+m},{m+2}) = {target}')
# Check each sum option


## Q21 — Derangements Ending with 1,2,3

The number of derangements of $1,2,3,4,5,6,7$ ending with $1,2,3$ in some order is:

- $3!(4! - 1)$
- $3! \cdot 4! / 2$
- $3! \cdot 4!$
- $3!(4! - 3!)$
- None of these

In [ ]:
from math import factorial
from discrete_tools import derangements_count, get_derangements
# Count derangements of 1..7 where last 3 positions hold 1,2,3 in some order
# The first 4 positions hold 4,5,6,7 (a derangement of 4,5,6,7)
# The last 3 positions hold 1,2,3 in some order where position i != i


## Q22 — System of Congruences (CRT)

$$x \equiv 1 \pmod{2}, \quad x \equiv 4 \pmod{5}, \quad x \equiv 3 \pmod{7}$$

The set of all solutions is:

- $\{12 + 70k \mid k \in \mathbb{Z}\}$
- $\{8 + 70k \mid k \in \mathbb{Z}\}$
- $\{1+2k\} \cup \{4+5k\} \cup \{3+7k\}$
- $\{70 + 12k \mid k \in \mathbb{Z}\}$
- $\{8 + 14k \mid k \in \mathbb{Z}\}$
- $\{59 + 70k \mid k \in \mathbb{Z}\}$
- None of these

In [ ]:
# congruences_system_solver([[1,2],[4,5],[3,7]])


## Q23 — Induction Proof Ordering (Multi-part)

Proof fragments for: *a simple graph on $n \geq 1$ vertices has at most $\binom{n}{2}$ edges.*

- **A.** Inductive step assuming for *every* $n \geq 1$ (strong induction style)
- **B.** Inductive step assuming for *some fixed* $n \geq 1$ (standard)
- **C.** Conclusion: follows by induction
- **D.** Bound: $v$ has $\leq n$ edges, so $G$ has $\leq n + \binom{n}{2} = \binom{n+1}{2}$
- **E.** Base case $n = 1$
- **F.** Let $G$ be a graph on $n+1$ vertices, pick vertex $v$, let $G' = G - v$
- **G.** Constructs $G'$ on $n+1$ vertices by adding a vertex to $G$ (wrong direction)

| Position | A | B | C | D | E | F | G | No more |
|---|---|---|---|---|---|---|---|---|
| First fragment | | | | | | | | |
| Second fragment | | | | | | | | |
| Third fragment | | | | | | | | |
| Fourth fragment | | | | | | | | |
| Fifth fragment | | | | | | | | |

In [ ]:
# Pure reasoning — no computation needed
# Think: base case first, then which induction hypothesis is correct,
# then the step, then the arithmetic bound, then the conclusion


## Q24 — Predicate Logic Translation

Let $P(x)$ mean "$x$ is prime". Which statement translates to:
$$\forall x \in \mathbb{Z}\; (x > 1 \to (\exists y \in \mathbb{N}\; (P(y) \wedge x < y < 2x)))$$

- For every integer $x > 1$, every number strictly between $x$ and $2x$ is prime.
- None of these
- For every integer $x$, either $x > 1$ or there is a prime $y$ with $x < y < 2x$.
- For any integer $x > 1$ and any prime $y$, if $x > 1$ then $x < y < 2x$.
- For every integer $x > 1$, there is a prime $y$ such that $x < y < 2x$.
- There exist infinitely many primes.

In [ ]:
# Pure logic — parse the quantifiers carefully
# Outer: forall x in Z
# Antecedent: x > 1
# Consequent: exists y in N, (P(y) AND x < y < 2x)


## Q25 — GCD Constraint

Let $a, b$ be positive integers with $ab = 5292 = 2^2 \cdot 3^3 \cdot 7^2$.

Which value **CANNOT** be $\gcd(a, b)$?

- $1$
- $3$
- $36$
- $42$
- All of these are possible

In [ ]:
import sympy as sp
n = 5292
print(sp.factorint(n))  # 2^2 * 3^3 * 7^2
# If gcd(a,b) = d, then a=d*a', b=d*b' with gcd(a',b')=1 and d^2*a'*b'=5292
# So d^2 must divide 5292
for d in [1, 3, 36, 42]:
    if 5292 % (d*d) == 0:
        print(f'd={d}: d^2={d**2} divides 5292? YES')
    else:
        print(f'd={d}: d^2={d**2} divides 5292? NO  <-- cannot be gcd')


## Q26 — Distinguishable Balls into Distinct Boxes (Multi-part)

Put $n+1$ distinguishable balls into $n$ distinct boxes.

| Scenario | None of these | $2n!$ | $\binom{n+1}{2} n!$ | $n^{n+1} - 2n!$ | $n^{n+1} - \binom{n+1}{2} n!$ |
|---|---|---|---|---|---|
| At least one box is empty — number of ways | | | | | |
| No box is empty — number of ways | | | | | |

In [ ]:
from math import comb, factorial
# Total ways (no restriction): n^(n+1)
# Exactly one box empty: choose which box is empty C(n,1), then the n+1 balls
#   go into n-1 boxes with no restrictions... think carefully
n = 3  # small example to verify
total = n**(n+1)
print(f'n={n}, total ways = {n}^{n+1} = {total}')
# Surjective (no empty box): inclusion-exclusion
no_empty = sum((-1)**k * comb(n, k) * (n-k)**(n+1) for k in range(n+1))
print(f'No box empty: {no_empty}')
print(f'At least one empty: {total - no_empty}')
print(f'C(n+1,2)*n! = {comb(n+1,2)*factorial(n)}')
